# Fase 3 - Majority Voting: LR + RF + SVM
Ensamble por mayoría de votos de los tres modelos ML entrenados en la Fase 2.
Para cada texto, la clase predicha es la que obtiene al menos 2 de 3 votos.
Se evalúa sobre el corpus combinado y por variante individual,
con representación TF-IDF y GloVe.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time
from scipy.stats import mode
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score, classification_report
)

## 2. Configuración de rutas y constantes

In [2]:
DATA_DIR   = '../data'
MODELS_DIR = '../data/modelos'
GLOVE_VEC  = '../glove-sbwc.i25.vec'

FEATURE_COLS = ['n_exc','n_int','n_may','n_emo','n_ris',
                'n_neg','n_elo','n_com','n_pun']

DATASETS_NOMBRES = ['combinado', 'mx', 'es', 'cu']

print('Configuración lista.')

Configuración lista.


## 3. Carga de datasets

In [3]:
df_test     = pd.read_csv(f'{DATA_DIR}/test_clean.csv')
df_test_mx  = pd.read_csv(f'{DATA_DIR}/test_clean_mx.csv')
df_test_es  = pd.read_csv(f'{DATA_DIR}/test_clean_es.csv')
df_test_cu  = pd.read_csv(f'{DATA_DIR}/test_clean_cu.csv')

TESTS = {
    'combinado': df_test,
    'mx':        df_test_mx,
    'es':        df_test_es,
    'cu':        df_test_cu,
}

print('Datasets de test cargados:')
for nombre, df in TESTS.items():
    print(f'  {nombre:12} {len(df):,} registros')

Datasets de test cargados:
  combinado    1,800 registros
  mx           600 registros
  es           600 registros
  cu           600 registros


## 4. Carga de vectores GloVe
Se reutiliza el mismo loader de la Fase 2.

In [4]:
class GloVeModel:
    def __init__(self, filepath):
        self.key_to_index = {}
        vectors_list = []
        expected_dim = None
        print('Cargando vectores GloVe...')
        t0 = time.time()
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i % 100000 == 0 and i > 0:
                    print(f'  {i:,} vectores...', end='\r')
                parts = line.rstrip().split(' ')
                if len(parts) < 2:
                    continue
                word = parts[0]
                try:
                    vector = np.array(parts[1:], dtype=np.float32)
                except ValueError:
                    continue
                if expected_dim is None:
                    expected_dim = len(vector)
                if len(vector) != expected_dim:
                    continue
                self.key_to_index[word] = len(vectors_list)
                vectors_list.append(vector)
        self.vectors     = np.vstack(vectors_list)
        self.vector_size = self.vectors.shape[1]
        print(f'\nListo en {time.time()-t0:.1f}s | '
              f'{len(self.key_to_index):,} vectores | '
              f'{self.vector_size} dimensiones')

    def __contains__(self, word):
        return word in self.key_to_index

    def __getitem__(self, word):
        return self.vectors[self.key_to_index[word]]

glove = GloVeModel(GLOVE_VEC)

Cargando vectores GloVe...
  800,000 vectores...
Listo en 33.8s | 1 vectores | 1 dimensiones


In [5]:
def vectorizar_glove(textos, modelo, dim=300):
    vectores = []
    for texto in textos:
        palabras = str(texto).split()
        vecs = [modelo[w] for w in palabras if w in modelo]
        vectores.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    return np.array(vectores)

## 5. Función de majority voting
Dadas las predicciones de 3 modelos, retorna la clase que obtiene al menos 2 votos.

In [6]:
def majority_vote(pred_lr, pred_rf, pred_svm):
    """
    Aplica majority voting sobre las predicciones de LR, RF y SVM.
    Retorna array con la clase que obtiene al menos 2 de 3 votos.
    """
    stacked = np.vstack([pred_lr, pred_rf, pred_svm]).T  # (n_samples, 3)
    votos, _ = mode(stacked, axis=1, keepdims=False)
    return votos.flatten()

def metricas(y_true, y_pred, nombre):
    return {
        'modelo':      nombre,
        'accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'f1_macro':    round(f1_score(y_true, y_pred, average='macro'), 4),
        'f1_ironico':  round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'p_ironico':   round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'r_ironico':   round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }

print('Funciones listas.')

Funciones listas.


## 6. Voting con representación TF-IDF

In [7]:
RESULTADOS_VOTING = []

for ds in DATASETS_NOMBRES:
    df_te = TESTS[ds]
    X_te  = df_te[['MESSAGE_CLEAN'] + FEATURE_COLS]
    y_te  = df_te['IS_IRONIC'].values

    # Cargar modelos TF-IDF
    lr  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_lr_{ds}.pkl')
    rf  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_rf_{ds}.pkl')
    svm = joblib.load(f'{MODELS_DIR}/ml_tf-idf_svm_{ds}.pkl')

    pred_lr  = lr.predict(X_te)
    pred_rf  = rf.predict(X_te)
    pred_svm = svm.predict(X_te)
    pred_vot = majority_vote(pred_lr, pred_rf, pred_svm)

    print(f'\n=== TF-IDF | {ds.upper()} ===')
    for nombre, pred in [('LR', pred_lr), ('RF', pred_rf),
                         ('SVM', pred_svm), ('VOTING', pred_vot)]:
        m = metricas(y_te, pred, nombre)
        m['dataset'] = ds
        m['repr']    = 'TF-IDF'
        RESULTADOS_VOTING.append(m)
        print(f'  {nombre:8} Accuracy={m["accuracy"]:.4f} '
              f'F1-Macro={m["f1_macro"]:.4f} '
              f'F1-Irónico={m["f1_ironico"]:.4f}')


=== TF-IDF | COMBINADO ===
  LR       Accuracy=0.7000 F1-Macro=0.6668 F1-Irónico=0.5617
  RF       Accuracy=0.6967 F1-Macro=0.6423 F1-Irónico=0.5027
  SVM      Accuracy=0.7011 F1-Macro=0.6681 F1-Irónico=0.5633
  VOTING   Accuracy=0.7006 F1-Macro=0.6665 F1-Irónico=0.5600

=== TF-IDF | MX ===
  LR       Accuracy=0.7167 F1-Macro=0.6796 F1-Irónico=0.5707
  RF       Accuracy=0.6633 F1-Macro=0.6284 F1-Irónico=0.5144
  SVM      Accuracy=0.7117 F1-Macro=0.6760 F1-Irónico=0.5686
  VOTING   Accuracy=0.7167 F1-Macro=0.6812 F1-Irónico=0.5750

=== TF-IDF | ES ===
  LR       Accuracy=0.7367 F1-Macro=0.7154 F1-Irónico=0.6376
  RF       Accuracy=0.7283 F1-Macro=0.6866 F1-Irónico=0.5722
  SVM      Accuracy=0.7417 F1-Macro=0.7227 F1-Irónico=0.6501
  VOTING   Accuracy=0.7383 F1-Macro=0.7180 F1-Irónico=0.6424

=== TF-IDF | CU ===
  LR       Accuracy=0.7133 F1-Macro=0.6640 F1-Irónico=0.5351
  RF       Accuracy=0.7167 F1-Macro=0.6594 F1-Irónico=0.5198
  SVM      Accuracy=0.7167 F1-Macro=0.6638 F1-Irónico=0

## 7. Voting con representación GloVe

In [8]:
for ds in DATASETS_NOMBRES:
    df_te = TESTS[ds]
    y_te  = df_te['IS_IRONIC'].values

    # Construir matriz GloVe
    glove_te = vectorizar_glove(df_te['MESSAGE_CLEAN'], glove)
    X_te = np.hstack([glove_te, df_te[FEATURE_COLS].values.astype(float)])

    # Cargar modelos GloVe
    lr  = joblib.load(f'{MODELS_DIR}/ml_glove_lr_{ds}.pkl')
    rf  = joblib.load(f'{MODELS_DIR}/ml_glove_rf_{ds}.pkl')
    svm = joblib.load(f'{MODELS_DIR}/ml_glove_svm_{ds}.pkl')

    pred_lr  = lr.predict(X_te)
    pred_rf  = rf.predict(X_te)
    pred_svm = svm.predict(X_te)
    pred_vot = majority_vote(pred_lr, pred_rf, pred_svm)

    print(f'\n=== GloVe | {ds.upper()} ===')
    for nombre, pred in [('LR', pred_lr), ('RF', pred_rf),
                         ('SVM', pred_svm), ('VOTING', pred_vot)]:
        m = metricas(y_te, pred, nombre)
        m['dataset'] = ds
        m['repr']    = 'GloVe'
        RESULTADOS_VOTING.append(m)
        print(f'  {nombre:8} Accuracy={m["accuracy"]:.4f} '
              f'F1-Macro={m["f1_macro"]:.4f} '
              f'F1-Irónico={m["f1_ironico"]:.4f}')


=== GloVe | COMBINADO ===
  LR       Accuracy=0.6461 F1-Macro=0.5913 F1-Irónico=0.4417
  RF       Accuracy=0.6456 F1-Macro=0.5920 F1-Irónico=0.4443
  SVM      Accuracy=0.6511 F1-Macro=0.5942 F1-Irónico=0.4423
  VOTING   Accuracy=0.6494 F1-Macro=0.5940 F1-Irónico=0.4441

=== GloVe | MX ===
  LR       Accuracy=0.6133 F1-Macro=0.5692 F1-Irónico=0.4314
  RF       Accuracy=0.6367 F1-Macro=0.5923 F1-Irónico=0.4577
  SVM      Accuracy=0.6050 F1-Macro=0.5605 F1-Irónico=0.4205
  VOTING   Accuracy=0.6150 F1-Macro=0.5695 F1-Irónico=0.4296

=== GloVe | ES ===
  LR       Accuracy=0.5850 F1-Macro=0.5801 F1-Irónico=0.5346
  RF       Accuracy=0.5900 F1-Macro=0.5752 F1-Irónico=0.4959
  SVM      Accuracy=0.5850 F1-Macro=0.5804 F1-Irónico=0.5363
  VOTING   Accuracy=0.5850 F1-Macro=0.5804 F1-Irónico=0.5363

=== GloVe | CU ===
  LR       Accuracy=0.6850 F1-Macro=0.6087 F1-Irónico=0.4358
  RF       Accuracy=0.6567 F1-Macro=0.6118 F1-Irónico=0.4798
  SVM      Accuracy=0.6883 F1-Macro=0.6114 F1-Irónico=0.438

## 8. Tabla comparativa: modelos individuales vs Voting

In [9]:
df_vot = pd.DataFrame(RESULTADOS_VOTING)[
    ['dataset','repr','modelo','accuracy','f1_macro','f1_ironico','p_ironico','r_ironico']
]

print('TABLA COMPLETA: MODELOS INDIVIDUALES vs MAJORITY VOTING')
print(df_vot.to_string(index=False))

# Resumen: ¿el voting mejora al mejor individual?
print('\n\nRESUMEN: ¿Voting mejora al mejor modelo individual?')
print('-' * 65)
for repr_nombre in ['TF-IDF', 'GloVe']:
    for ds in DATASETS_NOMBRES:
        subset = df_vot[(df_vot['dataset']==ds) & (df_vot['repr']==repr_nombre)]
        mejor_ind = subset[subset['modelo'] != 'VOTING']['f1_macro'].max()
        voting    = subset[subset['modelo'] == 'VOTING']['f1_macro'].values[0]
        delta     = voting - mejor_ind
        signo     = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
        print(f'  {repr_nombre:8} {ds:12} '
              f'mejor_ind={mejor_ind:.4f} '
              f'voting={voting:.4f} '
              f'{signo} {delta:+.4f}')

df_vot.to_csv(f'{DATA_DIR}/resultados_voting.csv', index=False)
print(f'\nGuardado: {DATA_DIR}/resultados_voting.csv')

TABLA COMPLETA: MODELOS INDIVIDUALES vs MAJORITY VOTING
  dataset   repr modelo  accuracy  f1_macro  f1_ironico  p_ironico  r_ironico
combinado TF-IDF     LR    0.7000    0.6668      0.5617     0.5466     0.5776
combinado TF-IDF     RF    0.6967    0.6423      0.5027     0.5531     0.4608
combinado TF-IDF    SVM    0.7011    0.6681      0.5633     0.5482     0.5793
combinado TF-IDF VOTING    0.7006    0.6665      0.5600     0.5479     0.5726
       mx TF-IDF     LR    0.7167    0.6796      0.5707     0.5736     0.5678
       mx TF-IDF     RF    0.6633    0.6284      0.5144     0.4931     0.5377
       mx TF-IDF    SVM    0.7117    0.6760      0.5686     0.5644     0.5729
       mx TF-IDF VOTING    0.7167    0.6812      0.5750     0.5721     0.5779
       es TF-IDF     LR    0.7367    0.7154      0.6376     0.5890     0.6950
       es TF-IDF     RF    0.7283    0.6866      0.5722     0.6022     0.5450
       es TF-IDF    SVM    0.7417    0.7227      0.6501     0.5926     0.7200
       e

## 9. Reporte detallado del mejor voting

In [10]:
# Reporte completo del mejor resultado de voting (TF-IDF combinado)
df_te = TESTS['combinado']
X_te  = df_te[['MESSAGE_CLEAN'] + FEATURE_COLS]
y_te  = df_te['IS_IRONIC'].values

lr  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_lr_combinado.pkl')
rf  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_rf_combinado.pkl')
svm = joblib.load(f'{MODELS_DIR}/ml_tf-idf_svm_combinado.pkl')

pred_vot = majority_vote(
    lr.predict(X_te),
    rf.predict(X_te),
    svm.predict(X_te)
)

print('=== Classification Report: Voting TF-IDF Combinado ===')
print(classification_report(y_te, pred_vot, target_names=['No irónico','Irónico']))

=== Classification Report: Voting TF-IDF Combinado ===
              precision    recall  f1-score   support

  No irónico       0.78      0.76      0.77      1201
     Irónico       0.55      0.57      0.56       599

    accuracy                           0.70      1800
   macro avg       0.66      0.67      0.67      1800
weighted avg       0.70      0.70      0.70      1800

